<a href="https://colab.research.google.com/github/machinelearnerme/DV_Assignment_Dashboard/blob/main/DV_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# confectionery_dashboard_full.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from io import BytesIO
from streamlit_plotly_events import plotly_events

# ===============================
# PAGE CONFIG
# ===============================
st.set_page_config(page_title="Confectionary Sales Dashboard (UK)", layout="wide")
st.title("🍬 Confectionary Sales Dashboard (UK)")
st.markdown("""
Interactive dashboard visualising confectionary sales, profitability, and regional trends in the UK.
Use filters on the left to explore specific regions, products, and date ranges.
""")

# ===============================
# FILE UPLOAD
# ===============================
file_path = "./Confectionary [4564].xlsx"

if file_path:
    # --- Load and clean data ---
    df = pd.read_excel(file_path)
    df = df.rename(columns={'Profit(£)': 'Sell(£)', 'Revenue(£)': 'Total Profit(£)'})
    df = df.dropna(subset=["Units Sold"])
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.strftime("%b")
    df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
    df["Half"] = df["Date"].dt.month.map(lambda x: "H1" if x <= 6 else "H2")
    df["Half-Year"] = df["Year"].astype(str) + "-" + df["Half"]
    # ================================
    # Standarized Confectionary Column
    #=================================

    df["Confectionary"] = df["Confectionary"].replace({
        "Choclate Chunk": "Chocolate Chunk",
        "Caramel nut" : "Caramel Nut"
    })
    # --- Profit & revenue calculations ---
    df["Profit per Unit (£)"] = df["Sell(£)"] - df["Cost(£)"]
    df["Profit (%)"] = ((df["Profit per Unit (£)"] / df["Cost(£)"]) * 100).round(2)
    df["Profit (%)"] = df.groupby("Confectionary")["Profit (%)"].transform(lambda x: x.fillna(x.median()))

    # ===============================
    # Handle missing Cost/Sell values using Profit(%)
    # ===============================
    df["Cost(£)"] = df["Cost(£)"].fillna(df["Sell(£)"] / (1 + df["Profit (%)"] / 100))
    df["Sell(£)"] = df["Sell(£)"].fillna(df["Cost(£)"] * (1 + df["Profit (%)"] / 100))
    df = df[~(df["Cost(£)"].isna() & df["Sell(£)"].isna())].reset_index(drop=True)

    # --- Derived calculations ---
    df["Total Profit (£)"] = (df["Sell(£)"] - df["Cost(£)"]) * df["Units Sold"]
    df["Revenue(£)"] = df["Sell(£)"] * df["Units Sold"]
    df["Profit per Unit (£)"] = df["Sell(£)"] - df["Cost(£)"]

    # ===============================
    # Display columns with missing values
    # ===============================
    missing_values = df.isna().sum()
    missing_columns = missing_values[missing_values > 0].sort_values(ascending=False)

    if not missing_columns.empty:
        st.warning("⚠️ The following columns have missing values:")
        st.dataframe(missing_columns)
    else:
        st.success("✅ No missing values found in the dataset.")

    # ===============================
    # SIDEBAR FILTERS
    # ===============================
    st.sidebar.header("🔍 Filters")
    country_filter = st.sidebar.multiselect("Select Region(s):", df["Country(UK)"].unique(), default=df["Country(UK)"].unique())
    product_filter = st.sidebar.multiselect("Select Confectionary:", df["Confectionary"].unique(), default=df["Confectionary"].unique())
    min_date, max_date = df["Date"].min(), df["Date"].max()
    start_date, end_date = st.sidebar.date_input("Select Date Range:", [min_date, max_date])

    # Apply filters
    df_filtered = df[
        (df["Country(UK)"].isin(country_filter)) &
        (df["Confectionary"].isin(product_filter)) &
        (df["Date"].between(pd.to_datetime(start_date), pd.to_datetime(end_date)))
    ]

    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]




    # ===============================
    # DASHBOARD TABS
    # ===============================
    tab0,tab1, tab2, tab3, tab4, tab5, tab6, tab7 = st.tabs([
        "📌 Key Performance Indicators (KPIs)",
        "🏴 Regional Performance",
        "🍫 Product Analysis",
        "📅 Sales & Profit Trends",
        "💰 Profitability Insights",
        "🗺️ UK Profit Map",
        "📈 Peak Month & Heatmap",
        "📊 Additional Insights"
    ])

    # (all other tab logic remains exactly the same as your current version)
    # Tabs include Regional Performance, Product Analysis, Trends, Profit Map, Heatmap, etc.

    # ===============================
    # TAB 0: DYNAMIC KPI CARDS
    # ===============================
    with tab0:
        st.header("📌 Key Performance Indicators (KPIs)")
        top_product = df_filtered.groupby("Confectionary")["Total Profit (£)"].sum().idxmax()
        top_product_value = df_filtered.groupby("Confectionary")["Total Profit (£)"].sum().max()
        top_region = df_filtered.groupby("Country(UK)")["Total Profit (£)"].sum().idxmax()
        top_region_value = df_filtered.groupby("Country(UK)")["Total Profit (£)"].sum().max()
        peak_month = df_filtered.groupby("Month")["Total Profit (£)"].sum().idxmax()
        peak_month_value = df_filtered.groupby("Month")["Total Profit (£)"].sum().max()
        top_margin_product = df_filtered.groupby("Confectionary")["Profit (%)"].mean().idxmax()
        top_margin_value = df_filtered.groupby("Confectionary")["Profit (%)"].mean().max()
        total_revenue = df_filtered["Revenue(£)"].sum()
        total_profit = df_filtered["Total Profit (£)"].sum()
        annual_avg_profit_margin = df_filtered.groupby("Year")["Profit (%)"].mean().mean()
        profit_to_revenue_ratio = (df_filtered["Total Profit (£)"].sum() / df_filtered["Revenue(£)"].sum()) * 100

        col1, col2, col3, col4, col7, col8 = st.columns(6)
        col1.metric("🏆 Top Product", f"{top_product}", f"£{top_product_value:,.0f}")
        col2.metric("🌍 Top Region", f"{top_region}", f"£{top_region_value:,.0f}")
        col3.metric("📅 Peak Month", f"{peak_month}", f"£{peak_month_value:,.0f}")
        col4.metric("💹 Top Avg Profit Margin", f"{top_margin_product}", f"{top_margin_value:.2f}%")
        col7.metric("📈 Avg Annual Profit Margin", f"{annual_avg_profit_margin:.2f}%")
        col8.metric("💱 Profit-to-Revenue Ratio", f"{profit_to_revenue_ratio:.2f}%")


        st.divider()

        # ===============================
        # OVERVIEW METRICS
        # ===============================
        st.header("📊 Overview Metrics")
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Total Units Sold", f"{df_filtered['Units Sold'].sum():,}")
        col2.metric("Total Revenue (£)", f"{df_filtered['Revenue(£)'].sum():,.2f}")
        col3.metric("Total Profit (£)", f"{df_filtered['Total Profit (£)'].sum():,.2f}")
        col4.metric("Average Profit Margin (%)", f"{df_filtered['Profit (%)'].mean():.2f}")
        st.divider()

        # ===============================
        # NEW PLOT: Total Revenue by Region
        # ===============================
        st.subheader("💰 Total Revenue by Region")
        region_revenue = df_filtered.groupby("Country(UK)")["Revenue(£)"].sum().reset_index().sort_values(by="Revenue(£)", ascending=False)
        revenue_chart = px.bar(
            region_revenue, x="Country(UK)", y="Revenue(£)", text="Revenue(£)",
            color="Country(UK)", title="Total Revenue by Region",
            labels={"Country(UK)": "Region", "Revenue(£)": "Total Revenue (£)"}
        )
        revenue_chart.update_traces(texttemplate="£%{text:,.0f}", textposition="outside", cliponaxis=False)
        revenue_chart.update_layout(
            showlegend=False, plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
            title_font=dict(size=20, color="darkblue", family="Arial Black")
        )
        st.plotly_chart(revenue_chart, use_container_width=True)

        st.divider()
    # ---------------------------
    # TAB 1: REGIONAL PERFORMANCE
    # ---------------------------
    with tab1:
        st.subheader("Regional Profit & Margin")
        region_profit = df_filtered.groupby("Country(UK)")["Total Profit (£)"].sum().reset_index()
        region_chart = px.bar(
            region_profit,
            x="Country(UK)",
            y="Total Profit (£)",
            color="Total Profit (£)",
            color_continuous_scale="Viridis",
            title="Total Profit by Region",
            text="Total Profit (£)",
            hover_data={"Country(UK)": True, "Total Profit (£)": ":,.0f"}
        )
        region_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)


        region_margin = df_filtered.groupby("Country(UK)")["Profit (%)"].mean().reset_index()
        margin_chart = px.bar(
            region_margin.sort_values("Profit (%)", ascending=True),
            x="Profit (%)",
            y="Country(UK)",
            color="Profit (%)",
            orientation="h",
            color_continuous_scale="RdYlGn",
	    title = "Profit (%) by Region",
            text="Profit (%)",
            hover_data={"Profit (%)": ":.2f"}
        )
        margin_chart.update_traces(texttemplate='%{text:.2f}%', textposition='outside', cliponaxis=False)

        col1, col2 = st.columns(2)
        col1.plotly_chart(region_chart, use_container_width=True)
        col2.plotly_chart(margin_chart, use_container_width=True)

        # Optional donut chart
        donut_chart = px.pie(
            region_margin,
            values="Profit (%)",
            names="Country(UK)",
            hole=0.4,
            title="Profit Margin Distribution by Region"
        )
        donut_chart.update_traces(textinfo='percent+label', hovertemplate="%{label}: %{value:.2f}%")
        col1, col2 = st.columns(2)
        col1.plotly_chart(donut_chart, use_container_width=True)




    # ---------------------------
    # TAB 2: PRODUCT ANALYSIS
    # ---------------------------
    with tab2:
        st.subheader("Confectionary Performance")
        product_profit = df_filtered.groupby("Confectionary")["Total Profit (£)"].sum().reset_index()
        product_chart = px.bar(
            product_profit,
            x="Confectionary",
            y="Total Profit (£)",
            color="Total Profit (£)",
            color_continuous_scale="Plasma",
            text="Total Profit (£)",
            title="Total Profit by Confectionary Type",
            hover_data={"Confectionary": True, "Total Profit (£)": ":,.0f"}
        )
        product_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)


        #sales_share = df_filtered.groupby("Confectionary")["Revenue(£)"].sum().reset_index()

        #pie_chart = px.pie(sales_share, names="Confectionary", values="Revenue(£)",title="Sales Share by Confectionary", hover_data={"Revenue(£)": ":,.0f"}        )

        #pie_chart.update_traces(textinfo="percent+label", hovertemplate="%{label}: £%{value:,.0f} (%{percent})")

        #col1, col2 = st.columns(2)
        #col1.plotly_chart(product_chart, use_container_width=True)
        #col2.plotly_chart(pie_chart, use_container_width=True)
        #======================================================================================================

        # Enable click selection
        product_chart.update_layout(clickmode='event+select')

        # ---- SALES PIE CHART ----
        sales_share = df_filtered.groupby("Confectionary")["Revenue(£)"].sum().reset_index()

        pie_chart = px.pie(
            sales_share,
            names="Confectionary",
            values="Revenue(£)",
            title="Sales Share by Confectionary",
            hover_data={"Revenue(£)": ":,.0f"},
        )

        pie_chart.update_traces(
            textinfo="percent+label",
            hovertemplate="%{label}: £%{value:,.0f} ",
            pull=[0.05] * len(sales_share),          )

        # Enable click selection on the pie
        pie_chart.update_layout(clickmode='event+select')

        # ---- DISPLAY SIDE BY SIDE ----
        col1, col2 = st.columns(2)
        col1.plotly_chart(product_chart, use_container_width=True)
        col2.plotly_chart(pie_chart, use_container_width=True)


    # ---------------------------
    # TAB 3: SALES & PROFIT TRENDS
    # ---------------------------
    with tab3:
        st.subheader("Sales & Profit Trends")

        # Monthly Sales Trend
        monthly_trend = df_filtered.groupby("Month")["Units Sold"].sum().reindex(month_order).reset_index()
        monthly_sales_chart = px.area(monthly_trend, x="Month", y="Units Sold", title="Average Monthly Sales Trend (All Regions)")

        # Quarterly Sales & Profit
        quarterly_sales = df_filtered.groupby(["Quarter", "Country(UK)"])["Units Sold"].sum().reset_index()
        quarterly_sales_chart = px.bar(
            quarterly_sales, x="Quarter", y="Units Sold", color="Country(UK)", barmode="group",
            title="Quarterly Sales by Region", text="Units Sold"
        )
        quarterly_sales_chart.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)

        quarterly_profit = df_filtered.groupby(["Quarter", "Country(UK)"])["Total Profit (£)"].sum().reset_index()
        quarterly_profit_chart = px.bar(
            quarterly_profit, x="Quarter", y="Total Profit (£)", color="Country(UK)", barmode="group",
            title="Quarterly Profit by Region", text="Total Profit (£)"
        )
        quarterly_profit_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)

        # Half-Yearly Sales & Profit
        half_sales = df_filtered.groupby(["Half-Year", "Country(UK)"])["Units Sold"].sum().reset_index()
        half_sales_chart = px.bar(
            half_sales, x="Half-Year", y="Units Sold", color="Country(UK)", barmode="group",
            text="Units Sold", title="Half-Yearly Sales by Region"
        )
        half_sales_chart.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)

        half_profit = df_filtered.groupby(["Half-Year", "Country(UK)"])["Total Profit (£)"].sum().reset_index()
        half_profit_chart = px.bar(
            half_profit, x="Half-Year", y="Total Profit (£)", color="Country(UK)", barmode="group",
            text="Total Profit (£)", title="Half-Yearly Profit by Region"
        )
        half_profit_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)

        # Yearly Sales & Profit

        yearly_sales = df_filtered.groupby(["Year", "Country(UK)"])["Units Sold"].sum().reset_index()
        yearly_sales_chart = px.bar(
            yearly_sales, x="Year", y="Units Sold", color="Country(UK)", barmode="group",
            text="Units Sold", title="Yearly Sales by Region"
        )
        yearly_sales_chart.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)

        yearly_profit = df_filtered.groupby(["Year", "Country(UK)"])["Total Profit (£)"].sum().reset_index()
        yearly_profit_chart = px.bar(
            yearly_profit, x="Year", y="Total Profit (£)", color="Country(UK)", barmode="group",
            title="Yearly Profit by Region", text="Total Profit (£)"
        )
        yearly_profit_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)

        # Display charts
        st.plotly_chart(monthly_sales_chart, use_container_width=True)
        col1, col2 = st.columns(2)
        col1.plotly_chart(quarterly_sales_chart, use_container_width=True)
        col2.plotly_chart(quarterly_profit_chart, use_container_width=True)
        col3, col4 = st.columns(2)
        col3.plotly_chart(half_sales_chart, use_container_width=True)
        col4.plotly_chart(half_profit_chart, use_container_width=True)
        col5, col6 = st.columns(2)
        col5.plotly_chart(yearly_sales_chart, use_container_width=True)
        col6.plotly_chart(yearly_profit_chart, use_container_width=True)

    # ========================================
    # TAB 4: PROFITABILITY INSIGHTS
    # ========================================
    with tab4:
        st.subheader("Profit Analysis")
        scatter = px.scatter(df_filtered, x="Sell(£)", y="Total Profit (£)", color="Confectionary", size="Units Sold", title="Selling Price vs Total Profit", hover_data=["Country(UK)"])
        hist = px.histogram(df_filtered, x="Profit (%)",nbins=60, color="Confectionary", title="Distribution of Profit Margins")
        hist.update_traces(texttemplate="%{y}", textposition="inside")
        #hist.update_layout(uniformtext_minsize=9, uniformtext_mode='hide',bargap=0.5, yaxis_title="Count", xaxis_title="Profit (%)", margin=dict(t=100))
        col1, col2 = st.columns(2)
        col1.plotly_chart(scatter, use_container_width=True)
        col2.plotly_chart(hist, use_container_width=True)

    # ========================================
    # TAB 5: UK PROFIT MAP
    # ========================================
    with tab5:
        st.subheader("Geographical Profit Distribution (UK Regions)")
        region_profit_map = df_filtered.groupby("Country(UK)")["Total Profit (£)"].sum().reset_index()
        uk_region_coords = {"England": [52.3555, -1.1743], "Scotland": [56.4907, -4.2026], "Wales": [52.1307, -3.7837], "N. Ireland": [54.7877, -6.4923], "Jersey": [49.2144, -2.1313]}
        region_profit_map["lat"] = region_profit_map["Country(UK)"].map(lambda x: uk_region_coords.get(x, [0,0])[0])
        region_profit_map["lon"] = region_profit_map["Country(UK)"].map(lambda x: uk_region_coords.get(x, [0,0])[1])
        map_chart = px.scatter_mapbox(region_profit_map, lat="lat", lon="lon", size="Total Profit (£)", color="Total Profit (£)", hover_name="Country(UK)", color_continuous_scale="Viridis", size_max=80, height=700,zoom=4,                                                      mapbox_style="carto-positron", title="Regional Profit Map")
        st.plotly_chart(map_chart, use_container_width=True)



    # ===============================
    # TAB 6: PEAK MONTH & HEATMAP
    # ===============================
    with tab6:
        st.subheader("Month-wise Peak Profit Analysis")

        # Monthly profit per confectionery & region
        monthly_profit = df_filtered.groupby(["Month", "Confectionary", "Country(UK)"])["Profit per Unit (£)"].mean().reset_index()
        monthly_profit["Month"] = pd.Categorical(monthly_profit["Month"], categories=month_order, ordered=True)
        peak_month_chart = px.bar(
            monthly_profit, x="Month", y="Profit per Unit (£)", color="Confectionary", barmode="group",
            facet_col="Country(UK)", title="Month-wise Average Profit per Confectionary and Region",
            hover_data={"Profit per Unit (£)": ":,.2f"}
        )
        st.plotly_chart(peak_month_chart, use_container_width=True)

        # Heatmap
        heatmap_data = df_filtered.groupby(["Month", "Country(UK)"])["Total Profit (£)"].sum().reset_index()
        heatmap_data["Month"] = pd.Categorical(heatmap_data["Month"], categories=month_order, ordered=True)
        heatmap_pivot = heatmap_data.pivot(index="Country(UK)", columns="Month", values="Total Profit (£)").fillna(0)
        heatmap_fig = go.Figure(data=go.Heatmap(
            z=heatmap_pivot.values,
            x=heatmap_pivot.columns,
            y=heatmap_pivot.index,
            text=heatmap_pivot.values.round(0),
            texttemplate="£%{text:,.0f}",
            colorscale='Viridis',
            colorbar=dict(title="Total Profit (£)"),
            hovertemplate='Region: %{y}<br>Month: %{x}<br>Total Profit: £%{z:,.0f}<extra></extra>'
        ))
        heatmap_fig.update_layout(title="💹 Month-wise Total Profit per Region", xaxis_title="Month", yaxis_title="Region", yaxis=dict(autorange="reversed"))
        st.plotly_chart(heatmap_fig, use_container_width=True)

        # Month-wise top confectionery
        month_product_profit = df_filtered.groupby(["Month", "Confectionary"])["Total Profit (£)"].sum().reset_index()
        month_product_profit["Month"] = pd.Categorical(month_product_profit["Month"], categories=month_order, ordered=True)
        month_top_product = month_product_profit.loc[month_product_profit.groupby("Month")["Total Profit (£)"].idxmax()]
        product_peak_chart = px.bar(
            month_top_product, x="Month", y="Total Profit (£)", color="Confectionary",
            text="Total Profit (£)", title="🏆 Month-wise Highest Profit by Confectionary",
            hover_data={"Total Profit (£)": ":,.0f"}
        )
        product_peak_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)
        st.plotly_chart(product_peak_chart, use_container_width=True)

        # Month-wise top region
        month_region_profit = df_filtered.groupby(["Month", "Country(UK)"])["Total Profit (£)"].sum().reset_index()
        month_region_profit["Month"] = pd.Categorical(month_region_profit["Month"], categories=month_order, ordered=True)
        month_top_region = month_region_profit.loc[month_region_profit.groupby("Month")["Total Profit (£)"].idxmax()]
        region_peak_chart = px.bar(
            month_top_region, x="Month", y="Total Profit (£)", color="Country(UK)",
            text="Total Profit (£)", title="🏆 Month-wise Highest Profit by Region",
            hover_data={"Total Profit (£)": ":,.0f"}
        )
        region_peak_chart.update_traces(texttemplate='£%{text:,.0f}', textposition='outside', cliponaxis=False)
        st.plotly_chart(region_peak_chart, use_container_width=True)

    # ---------------------------
    # TAB 7: ADDITIONAL INSIGHTS
    # ---------------------------
    with tab7:
        st.subheader("Additional Visual Insights")
        # Scatter Units Sold vs Profit per Unit
        scatter_chart = px.scatter(
            df_filtered, x="Units Sold", y="Profit per Unit (£)",
            color="Confectionary", size="Total Profit (£)",
            hover_data=["Country(UK)", "Revenue(£)"], title="Units Sold vs Profit per Unit"
        )
        st.plotly_chart(scatter_chart, use_container_width=True)

        # Pareto Analysis
        pareto_df = df_filtered.groupby("Confectionary")["Total Profit (£)"].sum().sort_values(ascending=False).reset_index()
        pareto_df["Cumulative Profit"] = pareto_df["Total Profit (£)"].cumsum()
        pareto_df["Cumulative %"] = pareto_df["Cumulative Profit"] / pareto_df["Total Profit (£)"].sum() * 100
        pareto_chart = px.line(
            pareto_df, x="Confectionary", y="Cumulative %", markers=True,
            title="Pareto Analysis: Cumulative Profit by Confectionary"
        )
        st.plotly_chart(pareto_chart, use_container_width=True)

        # Profit % distribution histogram
        hist_chart = px.histogram(
            df_filtered, x="Profit (%)", nbins=75, color ="Confectionary",
            title="Profit Margin (%) Distribution"
        )
#color_discrete_sequence=["green"]
        st.plotly_chart(hist_chart, use_container_width=True)

else:
    st.info("👆 Please upload your `Confectionary_4564.xlsx` file to begin exploring the dashboard.")

2026-03-18 15:32:27.893 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.895 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.896 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.897 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.898 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.900 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-18 15:32:27.900 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [5]:
!pip install streamlit_plotly_events


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 68.1 MB/s eta 0:00:00
